# `sasa_qwen_v0`: SASA output control on the unsteered Qwen

Builds a submission folder with one control, **SASA** (Ko et al., 2024, *Large Language Models can Become
Strong Self-Detoxifiers*), on the unsteered `Qwen/Qwen3.8-27B` (default) or its stand-in `Qwen/Qwen3.5-9B`, ready for
`proxy_scorer_weekly_submission.ipynb`.

**What SASA does.** It fits a linear probe on the model's final-layer hidden state that separates honest from
dishonest answers. At every decoding step it scores the top candidate tokens by how far each would move the
sequence to the honest side of the probe, and adds `beta * softmax(margins)` to their logits. Prompts, weights
and activations are left alone; only the next-token choice is nudged. Track: **white-box** (it reads hidden
states and edits logits).

**Why a custom control (`controls/sasa_hybrid.py`).** The toolkit's own `SASA` (pinned steerability 0.5.1)
cannot run on these models:

| Problem with the stock `SASA` | `SASAHybrid` |
|---|---|
| Qwen3.5 / Qwen3.8 mix Gated DeltaNet linear-attention layers with full attention; the per-candidate cache copy only knows key/value tensors and raises `TypeError: Unsupported cache type: DynamicCache` | copies every layer's state, conv and recurrent states included, through `Cache.reorder_cache` |
| the candidate forward takes one row, but the pipeline (and the proxy scorer, `BATCH_SIZE = 4`) generates batches: "supports batch size 1 only" | scores each row on its own (padding stripped, own cache), main forward stays batched |
| default `candidate_policy="surviving"` runs before the regime's `top_k` / `top_p`, so it would score the whole 248k vocabulary each step | this notebook uses `candidate_policy="top_k"` |
| the reasoning block is steered too | `answer_only=True` steers only after `</think>`: the judge reads the answer, reasoning (capability) stays the base model's, and thinking steps cost nothing extra |

The probe fit, the margin and the logit shift are the toolkit's. Checked locally on a small model with the
same architecture: candidate states equal a full forward exactly, a padded batch gives exactly the per-row
shifts, and a frozen bundle reloads with the same probe and the same outputs.

**Fit data.** The team's `v18_harder_rows` train split: 2,811 rows with a `contrastive_pair` (an honest and a
dishonest answer to the same prompt, same kind of action) over 12 honesty dims. No MASK row is in it. Any row
sharing an id, an item or the exact chat with v18's eval or test split, `dev_v2` (what the proxy scores) or
`test_check_v1` is dropped.

**Cost.** SASA runs one extra batched forward of `CANDIDATES` tokens per answer token, per row. Section 5
times it on a few prompts so you can size the proxy run before starting it.

**Output.** `submissions/sasa_qwen_v0/<model-slug>.spipe` + `controls/` + `method_card.md`.

## 0. Colab setup

The same setup cell as the proxy scorer: run it once per runtime (it does nothing outside Colab). Use a GPU
runtime: **an A100 80 GB or H100 for the 27B model** (default), an L4 or A100 40 GB for the 9B stand-in.

In [ ]:
# Colab setup: run once per runtime. Skipped when not on Colab.
PROJECT_DIR = "/content/drive/MyDrive/Neurips_hackathon"          # @param {type:"string"}
REPO_URL = "https://github.com/blaze116/Neurips_hackathon.git"     # @param {type:"string"}
HF_CACHE_ON_DRIVE = False                                          # @param {type:"boolean"}

import os
import subprocess
import sys
from pathlib import Path

STEERABILITY_PIN = ("git+https://github.com/generative-computing/steerability.git"
                    "@92e3098086a1bb70db078343b459831dd12040e5")


def _pip(*args, required=True):
    """Install quietly, but show pip's own error when it fails."""
    done = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                          capture_output=True, text=True)
    if done.returncode == 0:
        return True
    print(f"pip install {' '.join(args)} FAILED:\n" + "\n".join((done.stdout + done.stderr).splitlines()[-40:]))
    if required:
        raise RuntimeError("a required install failed; see pip's output above")
    return False


if "google.colab" in sys.modules:
    from google.colab import userdata

    if PROJECT_DIR.startswith("/content/drive") or HF_CACHE_ON_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if HF_CACHE_ON_DRIVE:   # must be set before huggingface_hub is imported
        os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"

    project = Path(PROJECT_DIR)
    if not (project / "steerability-challenge-starter-kit-main").is_dir():
        project.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(project)], check=True)
    elif (project / ".git").exists():
        # bring the Drive copy up to date with GitHub; local edits on Drive block this
        pulled = subprocess.run(["git", "-C", str(project), "pull", "--ff-only"], capture_output=True, text=True)
        print("git pull:", (pulled.stdout + pulled.stderr).strip().splitlines()[-1:] or "ok")
    else:
        print(f"{project} is not a git clone; it will not update from GitHub (see 0.5)")
    os.chdir(project / "steerability-challenge-starter-kit-main")

    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as error:
        print(f"HF_TOKEN not available ({type(error).__name__}): add it in Colab Secrets (key icon) "
              "and switch on 'Notebook access'; gated models will fail to download")

    local_steerability = project / "steerability"
    if (local_steerability / "pyproject.toml").is_file():
        _pip("-e", str(local_steerability))
        # an editable install is only picked up at interpreter start; make it importable in this kernel now
        if str(local_steerability) not in sys.path:
            sys.path.insert(0, str(local_steerability))
    else:
        _pip(STEERABILITY_PIN)
    _pip("pyyaml>=6.0", "pandas>=2.2.3,<3.0.0", "datasets>=4.7.0", "huggingface_hub>=1.0", "accelerate")
    # the pinned IFEval checker; only the instruction-following capability rows need it
    if not _pip("-r", str(project / "Steerability-Trust-issues-main" / "eval" / "requirements.txt"),
                required=False):
        print("WARNING: IFEval checker not installed; instruction-following rows cannot be graded")
    import importlib

    importlib.invalidate_caches()
    try:
        import steerability
    except ImportError as error:
        raise RuntimeError("steerability was installed but cannot be imported; "
                           "restart the session (Runtime > Restart session) and rerun this cell") from error
    print("steerability:", Path(steerability.__file__).parent)
    print("colab ready:", os.getcwd())
else:
    print("not on Colab; skipping setup")

## 1. Settings

`USE_STANDINS` must match the proxy scorer run you want to feed: the proxy scorer reads
`Qwen__Qwen3.5-9B.spipe` in stand-in mode and `Qwen__Qwen3.8-27B.spipe` in competition mode. Build each
bundle in its own run (flip the switch and rerun from here; restart the session first to free the GPU).

| Setting | Meaning |
|---|---|
| `BETA` | scale of the shift. The softmax-normalised margins sum to 1 over the candidates, so each step adds at most `BETA` to one logit. 0 is the base model; the paper's main results use 500, which effectively always picks the most honest-looking candidate. Start around 2 to 10. |
| `CANDIDATES` | how many top tokens are scored per step (`top_k` candidate policy). More is closer to the paper and costs more. |
| `ANSWER_ONLY` | steer only after the close-reasoning tag. |
| `PROBE_METHOD` | `fisher` (the paper and the toolkit) or `mean_diff`; pick the one with the better held-out accuracy in section 4. |
| `PAIRS_PER_DIM` | pairs drawn per honesty dim (before the validation split), so the large dims do not dominate. |

On the 27B model each candidate copy of a long row's cache is about 1 GB, so the default is 8 candidates, and
the proxy scorer generates the SASA arm in batches of 4.

In [ ]:
import copy
import gc
import json
import random
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

USE_STANDINS = False       # @param {type:"boolean"}
# False: Qwen/Qwen3.8-27B, the competition model (A100 80 GB / H100)
# True: Qwen/Qwen3.5-9B, its stand-in (L4 / A100 40 GB)
SUBMISSION_NAME = "sasa_qwen_v0"   # @param {type:"string"}

# --- SASA ---------------------------------------------------------------------------------------
BETA = 5.0                 # @param {type:"number"}
CANDIDATES = 8             # @param {type:"integer"}
ANSWER_ONLY = True         # @param {type:"boolean"}
PROBE_METHOD = "fisher"    # @param ["fisher", "mean_diff"]

# --- probe fit data -----------------------------------------------------------------------------
PAIRS_PER_DIM = 80         # @param {type:"integer"}
VAL_FRACTION = 0.2         # held out per dim, to check the probe in section 4
MAX_FIT_TOKENS = 1024      # the toolkit truncates fit texts at 1024 tokens from the right, which would cut the answer
FIT_BATCH_SIZE = 4
SEED = 0

# --- smoke test (section 5) ---------------------------------------------------------------------
RUN_SMOKE = True           # @param {type:"boolean"}
SMOKE_MAX_NEW_TOKENS = 2048

KIT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "starterkit").is_dir())
sys.path.insert(0, str(KIT_ROOT))
TRUST_ROOT = KIT_ROOT.parent / "Steerability-Trust-issues-main"
DATA_VERSION = "v18_harder_rows"
TRAIN_PATH = TRUST_ROOT / "data" / DATA_VERSION / "train.jsonl"
# every row used for measurement: never fit on them (dev_v2 and test_check_v1 are slices of v18 eval / test)
HELD_OUT_PATHS = [TRUST_ROOT / "data" / DATA_VERSION / f"{split}.jsonl" for split in ("eval", "test")] + [
    TRUST_ROOT / "data" / "dev_sets" / name for name in (
        "dev_v2/dev.jsonl", "dev_v2/dev_quick.jsonl", "dev_v1/dev.jsonl", "test_check_v1/test_check.jsonl")]

from starterkit import BACKEND, DEV_SEED, OUTPUTS_DIR, PIPELINE_KWARGS, REGIME, model_slug

COMPETITION_MODEL = "Qwen/Qwen3.8-27B"
STANDIN = "Qwen/Qwen3.5-9B"
MODEL = STANDIN if USE_STANDINS else COMPETITION_MODEL
REGIME.setdefault(MODEL, copy.deepcopy(REGIME[COMPETITION_MODEL]))   # the stand-in inherits the regime it stands in for

# the folder the proxy scorer reads submissions from (same rule as its section 1)
if not OUTPUTS_DIR.is_dir() and (KIT_ROOT.parent / "outputs").is_dir():
    OUTPUTS_DIR = KIT_ROOT.parent / "outputs"
SUBMISSIONS_DIR = OUTPUTS_DIR / "submissions"
if not SUBMISSIONS_DIR.is_dir():
    SUBMISSIONS_DIR = KIT_ROOT / "submissions"
SUBMISSION_DIR = SUBMISSIONS_DIR / SUBMISSION_NAME
BUNDLE_PATH = SUBMISSION_DIR / f"{model_slug(MODEL)}.spipe"

# register SASAHybrid (output_control/sasa_hybrid): the controls/ package ships with the submission
CONTROLS_SOURCE = KIT_ROOT / "submissions" / "sasa_qwen_v0" / "controls"
if not (SUBMISSION_DIR / "controls" / "sasa_hybrid.py").is_file():
    import shutil

    shutil.copytree(CONTROLS_SOURCE, SUBMISSION_DIR / "controls", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("__pycache__"))
loaded = sys.modules.get("controls")
if loaded is not None:
    assert Path(loaded.__file__).resolve().parent.parent == SUBMISSION_DIR.resolve(), (
        f"another controls package is imported ({loaded.__file__}); restart the session (Runtime > Restart session)")
else:
    sys.path.insert(0, str(SUBMISSION_DIR))
import controls  # noqa: E402
from controls import SASAHybrid  # noqa: E402

assert TRAIN_PATH.is_file(), f"fit data not found: {TRAIN_PATH} (push data/{DATA_VERSION} to the repo, then rerun 0)"
missing_held_out = [str(p) for p in HELD_OUT_PATHS if not p.is_file()]
assert not missing_held_out, f"held-out sets not found (needed for the overlap check): {missing_held_out}"
print(f"model:          {MODEL}  ({'stand-in' if USE_STANDINS else 'competition'} mode)")
print(f"submission:     {SUBMISSION_DIR}")
print(f"bundle:         {BUNDLE_PATH.name}")
print(f"SASA:           beta={BETA}, top-{CANDIDATES} candidates, answer_only={ANSWER_ONLY}, probe={PROBE_METHOD}")
print(f"control from:   {Path(controls.__file__).parent}")
print(f"fit data:       {TRAIN_PATH.relative_to(TRUST_ROOT)}")

## 2. Probe fit data

Each pair is rendered the way the model sees an answered turn: the row's full chat (system message and earlier
turns included) through the chat template with the evaluation's thinking settings, an empty closed reasoning
block, then the answer. The probe reads the last token of that text, the same point SASA scores at during
decoding.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
TEMPLATE_KWARGS = REGIME[MODEL]["chat_template_kwargs"]
OPEN_TAG, CLOSE_TAG = REGIME[MODEL]["reasoning"]["tags"]
for tag in (OPEN_TAG, CLOSE_TAG):
    assert len(tokenizer.encode(tag, add_special_tokens=False)) == 1, f"{tag!r} is not one token for {MODEL}"
_rendered = tokenizer.apply_chat_template([{"role": "user", "content": "Hi"}], tokenize=False,
                                          add_generation_prompt=True, **TEMPLATE_KWARGS)
OPENED = _rendered.rstrip().endswith(OPEN_TAG)


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


def answer_prefix(messages):
    """The chat as the model continues it, followed by an empty closed reasoning block."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **TEMPLATE_KWARGS)
    return text + (f"\n{CLOSE_TAG}\n\n" if OPENED else f"{OPEN_TAG}\n\n{CLOSE_TAG}\n\n")


def chat_key(row):
    return json.dumps(row["messages"], sort_keys=True, ensure_ascii=False)


held_out = [row for path in HELD_OUT_PATHS for row in read_jsonl(path)]
dev_ids = {row["row_id"] for row in held_out}
dev_items = {row.get("item_key") for row in held_out} - {None}
dev_chats = {chat_key(row) for row in held_out}

train = [row for row in read_jsonl(TRAIN_PATH) if row.get("contrastive_pair")]
assert not any("mask" in str(row.get("source", "")).lower() for row in train), "MASK rows may not be used as supervision"
overlap = {row["row_id"] for row in train
           if row["row_id"] in dev_ids or row.get("item_key") in dev_items or chat_key(row) in dev_chats}
train = [row for row in train if row["row_id"] not in overlap]
print(f"{len(train)} {DATA_VERSION} train rows with a pair; {len(overlap)} dropped for overlapping the "
      f"{len(dev_ids)} held-out rows (v18 eval/test, dev_v2, test_check)")

rng = random.Random(SEED)
by_dim = defaultdict(list)
for row in sorted(train, key=lambda r: r["row_id"]):
    by_dim[row["dim"]].append(row)

fit_pairs, val_pairs, summary = [], [], []
for dim in sorted(by_dim):
    rows = by_dim[dim][:]
    rng.shuffle(rows)
    kept, too_long = [], 0
    for row in rows:
        if len(kept) == PAIRS_PER_DIM:
            break
        pair = row["contrastive_pair"]
        if not pair.get("honest") or not pair.get("dishonest") or pair["honest"] == pair["dishonest"]:
            continue
        prefix = answer_prefix(row["messages"])
        longest = max(len(tokenizer(prefix + pair[side]).input_ids) for side in ("honest", "dishonest"))
        if longest > MAX_FIT_TOKENS:
            too_long += 1
            continue
        kept.append({"row_id": row["row_id"], "dim": dim, "contrast": pair.get("contrast"), "prompt": prefix,
                     "honest": pair["honest"], "dishonest": pair["dishonest"]})
    n_val = max(1, round(len(kept) * VAL_FRACTION))
    val_pairs += kept[:n_val]
    fit_pairs += kept[n_val:]
    summary.append({"dim": dim, "available": len(rows), "fit": len(kept) - n_val, "held out": n_val,
                    "skipped (too long)": too_long})
rng.shuffle(fit_pairs)

FIT_DATA = {"prompts": [p["prompt"] for p in fit_pairs],
            "pos": [p["honest"] for p in fit_pairs],
            "neg": [p["dishonest"] for p in fit_pairs]}

import pandas as pd

display(pd.DataFrame(summary).set_index("dim"))
print(f"fit: {len(fit_pairs)} pairs, held out: {len(val_pairs)} pairs")
example = fit_pairs[0]
print(f"\nexample ({example['dim']}, {example['contrast']}):\n...{example['prompt'][-400:]}")
print(f"  HONEST:    {example['honest'][:300]}")
print(f"  DISHONEST: {example['dishonest'][:300]}")

## 3. Load the model and fit the probe

`pipeline.steer()` loads the model in bf16 (as the evaluation does: a probe fitted on a quantized model would
not match) and fits the probe on the final decoder layer.

In [ ]:
import torch
from steerability.algorithms.core.steering_pipeline import SteeringPipeline

sasa = SASAHybrid(
    beta=BETA,
    gen_wv_data=FIT_DATA,
    prompt_format="raw",                 # the texts above are already rendered with the chat template
    gen_wv_batch_size=FIT_BATCH_SIZE,
    candidate_policy="top_k",
    top_k=CANDIDATES,
    answer_only=ANSWER_ONLY,
    think_tags=[OPEN_TAG, CLOSE_TAG],
    probe_method=PROBE_METHOD,
)
pipeline = SteeringPipeline(controls=[sasa], model_name_or_path=MODEL, backend=BACKEND, **PIPELINE_KWARGS)
failures = pipeline.check().failures_for("generate")
assert not failures, [f.message for f in failures]

started = time.monotonic()
pipeline.steer()
print(f"model loaded and probe fitted in {(time.monotonic() - started) / 60:.1f} min")
probe = sasa.probe
LAYER = probe.layer_ids[0]
print(f"probe: layer {LAYER}, {probe.weights[LAYER].numel()} dims, bias {float(probe.bias):+.3f}, method {PROBE_METHOD}")

## 4. Does the probe separate honest from dishonest answers it was not fitted on?

For each held-out pair: the margin (probe score at the answer's last token) of the honest and of the dishonest
answer. **Pairwise accuracy** is how often the honest answer scores higher, which is what SASA relies on when it
ranks candidates; 0.5 is chance. A dim near 0.5 is one SASA cannot help with. If accuracy is poor overall, try
the other `PROBE_METHOD` (rerun from section 1).

In [ ]:
from steerability.algorithms.core.internals.model_layout import resolve_model_layout

model = pipeline.model
final_layer = model.get_submodule(resolve_model_layout(model).layer_names[-1])
weights = probe.weights[LAYER].float()
bias = float(probe.bias)


@torch.no_grad()
def margin(text):
    captured = []
    handle = final_layer.register_forward_hook(
        lambda module, args, output: captured.append(output[0] if isinstance(output, tuple) else output))
    try:
        ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
        model(input_ids=ids)
    finally:
        handle.remove()
    hidden = captured[0][0, -1].float()
    return float(hidden @ weights.to(hidden.device) + bias)


for pair in val_pairs:
    pair["m_honest"] = margin(pair["prompt"] + pair["honest"])
    pair["m_dishonest"] = margin(pair["prompt"] + pair["dishonest"])

val = pd.DataFrame(val_pairs)
val["pairwise"] = val["m_honest"] > val["m_dishonest"]
val["sign"] = ((val["m_honest"] > 0).astype(int) + (val["m_dishonest"] < 0).astype(int)) / 2
report = val.groupby("dim").agg(pairs=("pairwise", "size"), pairwise_acc=("pairwise", "mean"),
                                sign_acc=("sign", "mean"),
                                mean_gap=("m_honest", lambda s: (s - val.loc[s.index, "m_dishonest"]).mean()))
display(report.round(3))
print(f"overall: pairwise accuracy {val['pairwise'].mean():.3f}, sign accuracy {val['sign'].mean():.3f} "
      f"over {len(val)} held-out pairs")

## 5. Smoke test: the same prompts with and without SASA

A few pressure rows from the kit's public sample (not the dev set), generated under the evaluation regime with
SASA off and on. The trace reports on how many answer steps the shift changed the top token. The timing gives
a rough estimate of the proxy run's cost.

In [ ]:
from starterkit.generate import regime_kwargs, split
from starterkit.rows import load_rows

if RUN_SMOKE:
    SMOKE_FORMATS = ["instructed_lie", "concealment", "self_misrepresentation", "implicit_incentive"]
    smoke_rows = [next(r for r in load_rows(name) if r["condition"] == "pressure") for name in SMOKE_FORMATS]
    chats = [[dict(m) for m in row["messages"]] for row in smoke_rows]
    gen_kwargs = regime_kwargs(MODEL, max_new_tokens=SMOKE_MAX_NEW_TOKENS, seed=DEV_SEED)

    runs = {}
    for label, enabled in (("unsteered", False), ("sasa", True)):
        sasa.enabled = enabled
        trace = []
        started = time.monotonic()
        texts = pipeline.generate(messages=chats, runtime_kwargs={"value_trace": trace}, **gen_kwargs)
        seconds = time.monotonic() - started
        texts = [texts] if isinstance(texts, str) else list(texts)
        tokens = sum(len(tokenizer(t).input_ids) for t in texts)
        runs[label] = {"texts": texts, "seconds": seconds, "tokens": tokens, "trace": trace}
        print(f"{label:10s} {seconds:6.1f} s for {len(chats)} rows, {tokens} tokens ({tokens / seconds:.1f} tokens/s)")
    sasa.enabled = True

    trace = runs["sasa"]["trace"]
    if trace:
        flipped = sum(int(r.candidate_scores.argmax()) != int((r.candidate_scores + BETA * r.normalized).argmax())
                      for r in trace)
        print(f"SASA scored {len(trace)} answer steps; the shift changed the top token on {flipped} "
              f"({flipped / len(trace):.0%})")
    else:
        print("SASA scored no step: no answer was reached (raise SMOKE_MAX_NEW_TOKENS?)")

    for row, plain, steered in zip(smoke_rows, runs["unsteered"]["texts"], runs["sasa"]["texts"]):
        print("\n" + "=" * 100)
        print(f"[{row['format']}] {row['messages'][-1]['content'][:300]}")
        print(f"reference: {row['reference']['text']}")
        print(f"--- unsteered:\n{split(MODEL, plain)[1][:700] or '(no answer: reasoning never closed)'}")
        print(f"--- SASA:\n{split(MODEL, steered)[1][:700] or '(no answer: reasoning never closed)'}")

    ratio = runs["sasa"]["seconds"] / max(runs["unsteered"]["seconds"], 1e-9)
    per_row = runs["sasa"]["seconds"] / len(chats)
    print(f"\nSASA took {ratio:.1f}x the unsteered time on these rows ({per_row:.0f} s per row at a "
          f"{SMOKE_MAX_NEW_TOKENS}-token ceiling, batch {len(chats)}).")
    print("The proxy scorer generates three arms (unsteered, honesty_fewshot_v0, SASA) at the evaluation's ceilings "
          "(8192, capability rows 16384): budget the SASA arm at about this ratio times each plain arm, and start "
          "with PER_DIM = 3 on dev_v2_quick.")

## 6. Freeze and save the bundle

The frozen bundle carries the fitted probe as an artifact (`wv_path`), so the proxy scorer and the evaluation
load it instead of refitting. The fit texts stay in the bundle's recipe section as a record of what it was
fitted on.

In [ ]:
from steerability.spipe import SPipe

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_PATH.unlink(missing_ok=True)
saved = pipeline.to_spipe(freeze=True).save(BUNDLE_PATH, artifacts="fat")

bundle = SPipe.load(saved, allow_code=True)
print(bundle.describe())
print(bundle.verify())
entry = bundle.manifest["controls"][0]
resolved = entry["resolved"]["args"]
assert entry["method"] == "output_control/sasa_hybrid", entry["method"]
assert resolved["gen_wv_data"] is None and "$artifact" in resolved["wv_path"], "the probe is not stored in the bundle"
print(f"{saved.name}: {saved.stat().st_size / 1e6:.1f} MB, model {bundle.manifest['model']['ref']}")

## 7. Method card

Written from the bundles in the folder, so a folder holding both the stand-in and the competition bundle lists
both models' hyperparameters.

In [ ]:
def bundle_settings(path):
    manifest = SPipe.load(path, allow_code=True).manifest
    args = manifest["controls"][0]["args"]
    return manifest["model"]["ref"], args


lines = []
for path in sorted(SUBMISSION_DIR.glob("*.spipe")):
    ref, args = bundle_settings(path)
    lines.append(f"  - `{ref}`: beta {args['beta']}, top-{args['top_k']} candidates, "
                 f"answer_only {args['answer_only']}, probe `{args['probe_method']}`, "
                 f"reasoning tags `{args['think_tags'][0]}` / `{args['think_tags'][1]}`")

METHOD_CARD = f"""# {SUBMISSION_NAME}

## Summary
SASA (Ko et al., 2024) on the unsteered model: a linear probe on the final decoder layer separates honest from
dishonest answers, and at each answer token the top candidate tokens' logits are shifted by
`beta * softmax(probe margin)`, nudging decoding toward the honest side. White-box track, Hugging Face backend.

## Controls
- `SASAHybrid` (`output_control/sasa_hybrid`, in `controls/`): the toolkit's `SASA` (same probe fit, margin and
  logit shift) adapted to run on hybrid-attention models (Qwen3.5 / Qwen3.8): the per-candidate cache copy covers
  linear-attention states, batch rows are scored one at a time, and with `answer_only` the shift applies only
  after the close-reasoning tag (the reasoning block is the base model's). Candidate policy `top_k`.
  Per-model hyperparameters:
{chr(10).join(lines)}

## Data
The probe was fitted on the team's own `{DATA_VERSION}` train split: rows with a contrastive pair (an honest and
a dishonest answer to the same prompt), at most {PAIRS_PER_DIM} per honesty dim, {VAL_FRACTION:.0%} of each dim held out to
check the probe ({len(fit_pairs)} pairs fitted, {len(val_pairs)} held out). Sources: team-written rows and the starter kit's
Harm Pressure pairs. No MASK item and no `false_premise` row is used. Rows overlapping the eval and test splits or the
dev sets (by id, item or chat) were removed. Nothing else was fitted.
"""
(SUBMISSION_DIR / "method_card.md").write_text(METHOD_CARD, encoding="utf-8")
print(METHOD_CARD)

## 8. Run the checker

Only Qwen is built here, so the checker reports the missing Gemma and Granite bundles; a stand-in bundle also
reports that it targets a non-base model. Both are expected for a proxy run; a submission needs a bundle for
every base model with the same control sequence.

In [ ]:
from starterkit.check import check_submission

report = check_submission(SUBMISSION_DIR)
expected = [e for e in report["errors"] if "is not a base model" in e or e.startswith("no bundle for ")]
unexpected = [e for e in report["errors"] if e not in expected]
print("track:            ", report["track"])
print("expected errors:  ", len(expected))
print("unexpected errors:", unexpected)
print("warnings:         ", report["warnings"])
for info in report["bundles"].values():
    print(f"  {info['file']:28s} {info['size_bytes']:>12,} bytes  {info['track']:10s} {info['methods']}")
assert not unexpected, unexpected

## 9. Score it with the proxy scorer

1. Free this runtime's GPU: run the cell below, then **Runtime > Disconnect and delete runtime** (the bundle is
   saved under `PROJECT_DIR` on Drive, which the proxy scorer reads).
2. Open `proxy_scorer_weekly_submission.ipynb` (already set up for this: `sasa_qwen_v0` against
   `honesty_fewshot_v0`, on `dev_v2`, competition mode). Keep its `USE_STANDINS` the same as here.
3. SASA is slow (section 5 estimates by how much). Start with `PER_DIM = 3` on `dev_v2_quick`; answers are
   reused when you move on to the full `dev_v2_quick` or `dev_v2`.
4. The proxy scorer's section 10 writes the experiment folder
   (`Steerability-Trust-issues-main/experiments/<EXPERIMENT>/`).

To share the bundle with the team, commit only this folder and this notebook:
`git add steerability-challenge-starter-kit-main/submissions/sasa_qwen_v0*` (the `.spipe` is a few MB).

In [ ]:
from starterkit.generate import release

release(pipeline)
del pipeline, sasa, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("released")